In [3]:
import sys
from pathlib import Path
import pandas as pd
import joblib
import numpy as np

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.lumenids.features import zeek_to_common

DATA_DIR = PROJECT_ROOT / "data" / "captures"
MODEL_DIR = PROJECT_ROOT / "models"

print("Project:", PROJECT_ROOT)
print("Capture folder exists:", DATA_DIR.exists())
print("Saved models:", sorted(p.name for p in MODEL_DIR.glob("rf_*.joblib")))

Project: C:\Users\Harman\lumen-ids
Capture folder exists: True
Saved models: ['rf_A_all11.joblib', 'rf_B_no_flags.joblib', 'rf_C_portable5.joblib', 'rf_baseline_v0.joblib']


In [4]:
for folder in sorted(DATA_DIR.glob("zeek_*")):
    log_path = folder / "conn.log"
    if log_path.exists():
        print(f"{folder.name}: {log_path.stat().st_size:,} bytes")

zeek_benign_01: 1,219 bytes
zeek_benign_02: 95,671 bytes
zeek_bruteforce_ssh_01: 1,111 bytes
zeek_dos_01: 380,030 bytes
zeek_portscan_01: 24,907,300 bytes
zeek_web_01: 37,276 bytes


In [9]:
from collections import Counter


model_names = ["A_all11", "B_no_flags", "C_portable5", "D_overlap4"]
models = {}

for name in model_names:
    path = MODEL_DIR / f"rf_{name}.joblib"
    if not path.exists():
        raise FileNotFoundError(f"Missing model: {path}")

    bundle = joblib.load(path)
    models[name] = bundle

results = []

for folder in sorted(DATA_DIR.glob("zeek_*")):
    log_path = folder / "conn.log"

    if not log_path.exists():
        continue

    conn = pd.read_json(log_path, lines=True)

    if len(conn) < 50:
        print(f"Skipping tiny capture: {folder.name} ({len(conn)} rows)")
        continue

    common = zeek_to_common(conn)

    print(f"\n{'=' * 50}")
    print(f"Capture: {folder.name} | Flows: {len(common)}")

    for name, bundle in models.items():
        model = bundle["model"]
        features = bundle["features"]

        missing = set(features) - set(common.columns)
        if missing:
            raise ValueError(f"{name} missing features: {missing}")

        predictions = model.predict(common[features])
        counts = Counter(predictions)

        benign_count = counts.get("benign", 0)
        attack_count = len(predictions) - benign_count

        results.append({
            "capture": folder.name,
            "model": name,
            "flows": len(predictions),
            "predicted_benign": benign_count,
            "predicted_attack": attack_count,
            "predicted_attack_pct": attack_count / len(predictions) * 100,
            "prediction_counts": dict(counts),
        })

        print(f"\n{name}:")
        print("Predictions:", dict(counts))
        print(f"Predicted attack share: {attack_count / len(predictions):.2%}")

results_df = pd.DataFrame(results)
display(results_df)

Skipping tiny capture: zeek_benign_01 (3 rows)

Capture: zeek_benign_02 | Flows: 235

A_all11:
Predictions: {'benign': 234, 'bruteforce': 1}
Predicted attack share: 0.43%

B_no_flags:
Predictions: {'benign': 235}
Predicted attack share: 0.00%

C_portable5:
Predictions: {'benign': 235}
Predicted attack share: 0.00%

D_overlap4:
Predictions: {'benign': 235}
Predicted attack share: 0.00%
Skipping tiny capture: zeek_bruteforce_ssh_01 (3 rows)

Capture: zeek_dos_01 | Flows: 1000

A_all11:
Predictions: {'dos': 556, 'benign': 444}
Predicted attack share: 55.60%

B_no_flags:
Predictions: {'benign': 648, 'dos': 352}
Predicted attack share: 35.20%

C_portable5:
Predictions: {'dos': 1000}
Predicted attack share: 100.00%

D_overlap4:
Predictions: {'dos': 1000}
Predicted attack share: 100.00%

Capture: zeek_portscan_01 | Flows: 65554

A_all11:
Predictions: {'benign': 65552, 'portscan': 1, 'dos': 1}
Predicted attack share: 0.00%

B_no_flags:
Predictions: {'benign': 65550, 'portscan': 3, 'dos': 1}
Pr

,capture,model,flows,predicted_benign,predicted_attack,predicted_attack_pct,prediction_counts
0,zeek_benign_02,A_all11,235,234,1,0.425532,"{'benign': 234, 'bruteforce': 1}"
1,zeek_benign_02,B_no_flags,235,235,0,0.000000,{'benign': 235}
2,zeek_benign_02,C_portable5,235,235,0,0.000000,{'benign': 235}
3,zeek_benign_02,D_overlap4,235,235,0,0.000000,{'benign': 235}
4,zeek_dos_01,A_all11,1000,444,556,55.600000,"{'dos': 556, 'benign': 444}"
5,zeek_dos_01,B_no_flags,1000,648,352,35.200000,"{'benign': 648, 'dos': 352}"
6,zeek_dos_01,C_portable5,1000,0,1000,100.000000,{'dos': 1000}
7,zeek_dos_01,D_overlap4,1000,0,1000,100.000000,{'dos': 1000}
8,zeek_portscan_01,A_all11,65554,65552,2,0.003051,"{'benign': 65552, 'portscan': 1, 'dos': 1}"
9,zeek_portscan_01,B_no_flags,65554,65550,4,0.006102,"{'benign': 65550, 'portscan': 3, 'dos': 1}"


In [6]:
cic = pd.read_pickle(PROJECT_ROOT / "data" / "processed" / "cic_common.pkl")

for fam in ["portscan", "dos", "benign"]:
    smp = cic[cic["family"] == fam].sample(
        n=min(2000, len(cic[cic["family"] == fam])),
        random_state=1,
    )

    for name, bundle in models.items():
        features = bundle["features"]
        predictions = bundle["model"].predict(smp[features])
        print(fam, name, round((predictions == fam).mean(), 3))
lab_log = DATA_DIR / "zeek_portscan_01" / "conn.log"
lab = zeek_to_common(pd.read_json(lab_log, lines=True))

ref = cic[cic["family"] == "portscan"]
features = models["A_all11"]["features"]

rows = []
for c in features:
    q5, q50, q95 = ref[c].quantile([0.05, 0.50, 0.95])

    rows.append({
        "feature": c,
        "CIC_p5": q5,
        "CIC_median": q50,
        "CIC_p95": q95,
        "lab_median": lab[c].median(),
        "lab_share_in_CIC_range": (
            ((lab[c] >= q5) & (lab[c] <= q95)).mean()
        ).round(3),
    })

range_df = pd.DataFrame(rows)
display(range_df)


portscan A_all11 1.0
portscan B_no_flags 1.0
portscan C_portable5 0.998
dos A_all11 1.0
dos B_no_flags 1.0
dos C_portable5 1.0
benign A_all11 0.998
benign B_no_flags 0.998
benign C_portable5 0.976


,feature,CIC_p5,CIC_median,CIC_p95,lab_median,lab_share_in_CIC_range
0,dst_port,135.00000,3546.0000,49155.0000,32761.500000,0.748
1,duration_us,11.00000,50.0000,97.0000,28555.512428,0.020
2,fwd_pkts,1.00000,1.0000,1.0000,1.000000,0.999
3,bwd_pkts,1.00000,1.0000,1.0000,1.000000,1.000
4,fwd_bytes,0.00000,0.0000,2.0000,0.000000,1.000
5,bwd_bytes,6.00000,6.0000,6.0000,0.000000,0.000
6,bytes_per_s,66666.66667,137931.0345,615384.6154,0.000000,0.000
7,pkts_per_s,20618.55670,40000.0000,181818.1818,63.754122,0.020
8,syn_count,0.00000,0.0000,0.0000,1.000000,0.000
9,rst_count,0.00000,0.0000,0.0000,1.000000,0.000


In [7]:
lab_log = DATA_DIR / "zeek_portscan_01" / "conn.log"
lab = zeek_to_common(pd.read_json(lab_log, lines=True))

ref = cic[cic["family"] == "portscan"]
features = models["A_all11"]["features"]

rows = []
for c in features:
    q5, q50, q95 = ref[c].quantile([0.05, 0.50, 0.95])

    rows.append({
        "feature": c,
        "CIC_p5": q5,
        "CIC_median": q50,
        "CIC_p95": q95,
        "lab_median": lab[c].median(),
        "lab_share_in_CIC_range": (
            ((lab[c] >= q5) & (lab[c] <= q95)).mean()
        ).round(3),
    })

range_df = pd.DataFrame(rows)
display(range_df)


,feature,CIC_p5,CIC_median,CIC_p95,lab_median,lab_share_in_CIC_range
0,dst_port,135.00000,3546.0000,49155.0000,32761.500000,0.748
1,duration_us,11.00000,50.0000,97.0000,28555.512428,0.020
2,fwd_pkts,1.00000,1.0000,1.0000,1.000000,0.999
3,bwd_pkts,1.00000,1.0000,1.0000,1.000000,1.000
4,fwd_bytes,0.00000,0.0000,2.0000,0.000000,1.000
5,bwd_bytes,6.00000,6.0000,6.0000,0.000000,0.000
6,bytes_per_s,66666.66667,137931.0345,615384.6154,0.000000,0.000
7,pkts_per_s,20618.55670,40000.0000,181818.1818,63.754122,0.020
8,syn_count,0.00000,0.0000,0.0000,1.000000,0.000
9,rst_count,0.00000,0.0000,0.0000,1.000000,0.000


In [11]:
import pandas as pd
from pathlib import Path

folder = Path("../data/MachineLearningCSV/TrafficLabelling")
matches = list(folder.glob("*Friday*PortScan*.csv"))

print("Files found:", len(matches))
for f in matches:
    print(f.name)

Files found: 1
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv


In [12]:
f = matches[0]
d = pd.read_csv(f, encoding="latin1")
d.columns = d.columns.str.strip()

print("File:", f)
print("Columns:", d.columns.tolist()[:12])
print("Rows:", len(d))
print("First 8 timestamps:", d["Timestamp"].head(8).tolist())

File: ..\data\MachineLearningCSV\TrafficLabelling\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Columns: ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets']
Rows: 286467
First 8 timestamps: ['7/7/2017 1:00', '7/7/2017 1:00', '7/7/2017 1:00', '7/7/2017 1:00', '7/7/2017 1:00', '7/7/2017 1:00', '7/7/2017 1:00', '7/7/2017 1:00']
